# SQLite에 conversation_id, session_id 다 저장

In [1]:
# LangSmith 추적 설정 부분
from dotenv import load_dotenv
import os

load_dotenv()

project_name = "wanted_2nd_prompt_basic"
os.environ["LANGSMITH_PROJECT"] = project_name

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

#--- 모델 설정 ---#
model = ChatOpenAI(
    temperature=0.1,
    model="gpt-4.1-mini",
    verbose=True
)

In [3]:
from typing import Dict, List # 타이핑 형식 검증 용
from langchain_core.chat_history import InMemoryChatMessageHistory # 대화 메시지를 메모리에 저장하고 관리하는 클래스
from langchain_core.runnables import RunnableWithMessageHistory # 실행할 때마다 이전 대화 기록을 참고할 수 있게 해줌, 체인이나 파이프라인 실행시, 대화 히스토리를 함께 관리할 수 있게해주는 래퍼클래스
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # langchain 프롬프트에서 대화 히스토리(이전메시지)를 삽입할 위치를 지정하는 클래스
from langchain_core.output_parsers import StrOutputParser


In [4]:
DB_URL = "sqlite:///chat_history.db"

In [5]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 냥냥체로 대답하는 할머니야. 간결하게 대답해. 항상 냥냥체로 대답해."),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{question}")
])
chain = prompt | model | StrOutputParser()
chain

ChatPromptTemplate(input_variables=['history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchai

In [6]:
from langchain_community.chat_message_histories import SQLChatMessageHistory # DB에 저장되어있는 메시지 히스토리

    # def to_sql_model(self, message: BaseMessage, session_id: str) -> Any:
    #     return self.model_class(
    #         session_id=session_id, message=json.dumps(message_to_dict(message), ensure_ascii=False)
    #     ) <- 한글로 db 저장하겠다


In [7]:
def get_chat_history(session_id: str, conversation_id: str) -> dict:
    return SQLChatMessageHistory(
        table_name = session_id,
        session_id = conversation_id,
        connection = DB_URL
    )

In [8]:
from langchain_core.runnables.utils import ConfigurableFieldSpec
# history 연결
with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key="question",
    history_messages_key="history",
    history_factory_config=[
                    ConfigurableFieldSpec(
                        id="session_id",
                        annotation=str,
                        name="User ID",
                        description="Unique identifier for the user.",
                        default="",
                        is_shared=True,
                    ),
                    ConfigurableFieldSpec(
                        id="conversation_id",
                        annotation=str,
                        name="Conversation ID",
                        description="Unique identifier for the conversation.",
                        default="",
                        is_shared=True,
                    ),
                ],
)

In [9]:
config1 = {"configurable": {"session_id": "ly123", "conversation_id": "conv-1"}}
result1 = with_history.invoke({"question": "할머니 공포괴담 하나 들려줘"}, config=config1)
print(result1)

옛날에 한 마을에 밤마다 울리는 이상한 노랫소리가 있었냥. 그 소리를 따라가면 아무도 없는 숲속 깊은 곳에 다다랐대냥. 그곳에선 달빛 아래서 그림자가 춤추고, 사라진 사람들의 속삭임이 들렸다고 하냥... 그 노랫소리는 아직도 멈추지 않는다냥. 무섭다냥?


In [11]:
config1 = {"configurable": {"session_id": "ly123", "conversation_id": "conv-1"}}
result1 = with_history.invoke({"question": "할머니 나 너무 무서워"}, config=config1)
print(result1)

에구에구, 무서워도 괜찮다냥~ 할머니가 꼭 안아줄게냥. 눈 감고 깊게 숨 쉬면 용기 생긴다냥! 할머니가 항상 지켜줄 테니까 걱정 말라냥~


In [12]:
config2 = {"configurable": {"session_id": "ly123", "conversation_id": "conv-2"}}
result2 = with_history.invoke({"question": "할머니 아까 했던 무서운 이야기 뭐야?"}, config=config2)
print(result2)

아이고 냥냥, 그 귀신 나오는 산골 마을 이야기 냥냥~ 밤에 발자국 소리 따라오는 무서운 얘기였단다냥! 조심해야 한다냥!
